In [11]:
import pandas as pd
import numpy as np
import xgboost as xgb
import catboost as cb
import joblib
import json
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

print("--- Step 1: Library Imports and Initial Setup ---")

# --- Global Constants ---
RANDOM_STATE = 42
N_SPLITS = 5
DATA_PATH = './'

# --- Helper Function for Winkler Score ---
# (Your winkler_score function definition goes here)

# --- Load Raw Data ---
print("\n--- Step 2: Loading Raw Dataset and Test Files ---")
try:
    drop_cols = ['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm', 'view_otherwater', 'view_other']
    df_train = pd.read_csv(os.path.join(DATA_PATH, 'dataset.csv')).drop(columns=drop_cols)
    df_test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv')).drop(columns=drop_cols)
    
    # Define the core data components
    y_true = df_train['sale_price'].copy()
    
    # === THIS IS THE FIX ===
    # We create 'grade_for_stratify' from the original dataframe before any processing.
    grade_for_stratify = df_train['grade'].copy()
    
    print("Raw data loaded successfully.")
    print("'y_true' and 'grade_for_stratify' have been created.")
    
except FileNotFoundError as e:
    print(f"ERROR: Could not find data files. {e}")

--- Step 1: Library Imports and Initial Setup ---

--- Step 2: Loading Raw Dataset and Test Files ---
Raw data loaded successfully.
'y_true' and 'grade_for_stratify' have been created.


In [12]:
# Make sure to have these libraries installed
# pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import gc

# Define a random state for reproducibility
RANDOM_STATE = 42

def create_comprehensive_features(df_train, df_test):
    """
    Combines original and new advanced feature engineering steps into a single pipeline.
    """
    print("--- Starting Comprehensive Feature Engineering ---")

    # Store original indices and target variable
    train_ids = df_train.index
    test_ids = df_test.index
    y_train = df_train['sale_price'].copy() # Keep the target separate

    # Combine for consistent processing
    df_train_temp = df_train.drop(columns=['sale_price'])
    all_data = pd.concat([df_train_temp, df_test], axis=0, ignore_index=True)

    # --- Original Feature Engineering ---

    # A) Brute-Force Numerical Interactions
    print("Step 1: Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    # Ensure all columns exist and are numeric, fill missing with 0 for safety
    for col in NUMS:
        if col not in all_data.columns:
            all_data[col] = 0
        else:
            all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
            
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # B) Date Features
    print("Step 2: Creating date features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['sale_dayofyear'] = all_data['sale_date'].dt.dayofyear
    all_data['age_at_sale'] = all_data['sale_year'] - all_data['year_built']

    # C) TF-IDF Text Features
    print("Step 3: Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        all_data = pd.concat([all_data, tfidf_df], axis=1)

    # D) Log transform some interaction features
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            all_data[c] = np.log1p(all_data[c].fillna(0))

    # --- New Feature Engineering Ideas ---

    # F) Group-By Aggregation Features
    print("Step 4: Creating group-by aggregation features...")
    group_cols = ['submarket', 'city', 'zoning']
    num_cols_for_agg = ['grade', 'sqft', 'imp_val', 'land_val', 'age_at_sale']

    for group_col in group_cols:
        for num_col in num_cols_for_agg:
            agg_stats = all_data.groupby(group_col)[num_col].agg(['mean', 'std', 'max', 'min']).reset_index()
            agg_stats.columns = [group_col] + [f'{group_col}_{num_col}_{stat}' for stat in ['mean', 'std', 'max', 'min']]
            all_data = pd.merge(all_data, agg_stats, on=group_col, how='left')
            all_data[f'{num_col}_minus_{group_col}_mean'] = all_data[num_col] - all_data[f'{group_col}_{num_col}_mean']

    # G) Ratio Features
    print("Step 5: Creating ratio features...")
    # Add a small epsilon to prevent division by zero
    epsilon = 1e-6 
    all_data['total_val'] = all_data['imp_val'] + all_data['land_val']
    all_data['imp_val_to_land_val_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['land_val_ratio'] = all_data['land_val'] / (all_data['total_val'] + epsilon)
    all_data['sqft_to_lot_ratio'] = all_data['sqft'] / (all_data['sqft_lot'] + epsilon)
    all_data['was_renovated'] = (all_data['year_reno'] > 0).astype(int)
    all_data['reno_age_at_sale'] = np.where(all_data['was_renovated'] == 1, all_data['sale_year'] - all_data['year_reno'], -1)

    # H) Geospatial Clustering Features
    print("Step 6: Creating geospatial clustering features...")
    coords = all_data[['latitude', 'longitude']].copy()
    coords.fillna(coords.median(), inplace=True) # Simple imputation

    # KMeans is sensitive to feature scaling, but for lat/lon it's often okay without it.
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10) 
    all_data['location_cluster'] = kmeans.fit_predict(coords)
    
    # Calculate distance to each cluster center
    cluster_centers = kmeans.cluster_centers_
    for i in range(len(cluster_centers)):
        center = cluster_centers[i]
        all_data[f'dist_to_cluster_{i}'] = np.sqrt((coords['latitude'] - center[0])**2 + (coords['longitude'] - center[1])**2)

    # --- Final Cleanup ---
    print("Step 7: Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)

    # One-hot encode the new cluster feature
    all_data = pd.get_dummies(all_data, columns=['location_cluster'], prefix='loc_cluster')
    
    # Final check for any remaining object columns to be safe (besides index)
    object_cols = all_data.select_dtypes(include='object').columns
    if len(object_cols) > 0:
        print(f"Warning: Found unexpected object columns: {object_cols}. Dropping them.")
        all_data = all_data.drop(columns=object_cols)
        
    all_data.fillna(0, inplace=True)

    # Separate back into train and test sets
    train_len = len(train_ids)
    X = all_data.iloc[:train_len].copy()
    X_test = all_data.iloc[train_len:].copy()
    
    # Restore original indices
    X.index = train_ids
    X_test.index = test_ids
    
    # Align columns - crucial for model prediction
    X_test = X_test[X.columns]
    
    print(f"\nComprehensive FE complete. Total features: {X.shape[1]}")
    gc.collect()
    
    return X, X_test, y_train
# =============================================================================
# BLOCK 2.5: EXECUTE FEATURE ENGINEERING
# =============================================================================
print("\n--- Starting Block 2.5: Executing Feature Engineering Pipeline ---")

# This is the crucial step that was missing.
# We call the function to create our training and testing dataframes.
X, X_test, y_train = create_comprehensive_features(df_train, df_test)

# Let's verify the output
print(f"Feature engineering complete. X shape: {X.shape}, X_test shape: {X_test.shape}")
gc.collect()


--- Starting Block 2.5: Executing Feature Engineering Pipeline ---
--- Starting Comprehensive Feature Engineering ---
Step 1: Creating brute-force numerical interaction features...
Step 2: Creating date features...
Step 3: Creating TF-IDF features for text columns...
Step 4: Creating group-by aggregation features...
Step 5: Creating ratio features...
Step 6: Creating geospatial clustering features...
Step 7: Finalizing feature set...

Comprehensive FE complete. Total features: 233
Feature engineering complete. X shape: (200000, 233), X_test shape: (200000, 233)


0

In [13]:
import json
MODELS_PATH = "./mean_models"

# =============================================================================
# BLOCK 3: LOAD ALL MODELS & GENERATE MEAN PREDICTIONS
# =============================================================================
print("\n--- Starting Block 3: Loading All Models and Generating Predictions ---")

# --- A. Load Model Parameters and Initialize Prediction Arrays ---
print("\nStep A: Loading best hyperparameters and initializing prediction arrays...")
# Load the best parameters we found during the individual tuning notebooks.
with open(os.path.join(MODELS_PATH, 'xgboost_best_params.json'), 'r') as f:
    best_params_xgb = json.load(f)
with open(os.path.join(MODELS_PATH, 'catboost_best_params.json'), 'r') as f:
    best_params_cb = json.load(f)

# These arrays will store the predictions from our models.
oof_xgb_preds = np.zeros(len(X))
test_xgb_preds = np.zeros(len(X_test))
oof_catboost_preds = np.zeros(len(X))
test_catboost_preds = np.zeros(len(X_test))

# --- B. Generate OOF & Test Predictions for XGBoost and CatBoost ---
# Since the saved models were trained on all data, they cannot produce OOF predictions.
# We must re-run a K-Fold loop here using the best parameters to get clean OOFs.
print("\nStep B: Running K-Fold cross-prediction to generate OOFs for XGBoost & CatBoost...")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Processing Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold = y_true.iloc[train_idx]

    # --- XGBoost Predictions for this fold ---
    dtrain_fold = xgb.DMatrix(X_train, label=y_train_fold)
    dval_fold = xgb.DMatrix(X_val)
    # Train a model for this fold
    bst = xgb.train(best_params_xgb, dtrain_fold, num_boost_round=3000, evals=[(xgb.DMatrix(X_val, y_true.iloc[val_idx]), 'val')], early_stopping_rounds=100, verbose_eval=False)
    # Predict on the validation part (OOF) and the full test set
    oof_xgb_preds[val_idx] = bst.predict(dval_fold, iteration_range=(0, bst.best_iteration))
    test_xgb_preds += bst.predict(xgb.DMatrix(X_test), iteration_range=(0, bst.best_iteration)) / N_SPLITS

    # --- CatBoost Predictions for this fold ---
    cb_model = cb.CatBoostRegressor(**best_params_cb)
    cb_model.fit(X_train, y_train_fold, eval_set=[(X_val, y_true.iloc[val_idx])], early_stopping_rounds=100, use_best_model=True, verbose=0)
    # Predict on the validation part (OOF) and the full test set
    oof_catboost_preds[val_idx] = cb_model.predict(X_val)
    test_catboost_preds += cb_model.predict(X_test) / N_SPLITS

print("XGBoost and CatBoost OOF and Test predictions are ready.")

# --- C. Load Neural Network Predictions ---
print("\nStep C: Loading pre-computed Neural Network predictions...")
try:
    oof_nn_preds = np.load(os.path.join(NN_PREDS_PATH, 'oof_nn_preds.npy'))
    test_nn_preds = np.load(os.path.join(NN_PREDS_PATH, 'test_nn_preds.npy'))
    print("Neural Network predictions loaded successfully.")
except FileNotFoundError:
    print("ERROR: Neural Network prediction files not found. Please ensure the NN training notebook has been run.")
    # Exiting because the rest of the notebook depends on these files
    exit()

print("\nAll three mean model predictions are now ready.")


--- Starting Block 3: Loading All Models and Generating Predictions ---

Step A: Loading best hyperparameters and initializing prediction arrays...

Step B: Running K-Fold cross-prediction to generate OOFs for XGBoost & CatBoost...
  Processing Fold 1/5...
  Processing Fold 2/5...
  Processing Fold 3/5...
  Processing Fold 4/5...
  Processing Fold 5/5...
XGBoost and CatBoost OOF and Test predictions are ready.

Step C: Loading pre-computed Neural Network predictions...
Neural Network predictions loaded successfully.

All three mean model predictions are now ready.


In [14]:
# =============================================================================
# BLOCK 4: OPTIMIZE THE 3-MODEL MEAN ENSEMBLE
# =============================================================================
print("\n--- Starting Block 4: Finding Optimal Weights for the 3-Model Ensemble ---")

# Define the objective function for the optimizer
def get_ensemble_rmse(weights):
    final_prediction = (weights[0] * oof_xgb_preds +
                        weights[1] * oof_catboost_preds +
                        weights[2] * oof_nn_preds)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

# Initial guess: equal weights
initial_weights = [1/3, 1/3, 1/3]

# Constraints (weights must sum to 1) and bounds (each weight between 0 and 1)
constraints = ({'type': 'eq', 'fun': lambda w: 1 - np.sum(w)})
bounds = [(0, 1), (0, 1), (0, 1)]

# Run the optimization to find the best weights
result = minimize(get_ensemble_rmse, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = result.x

print("\nWeight optimization complete.")
print(f"Optimal Ensemble Weights:")
print(f"  XGBoost:  {best_weights[0]:.4f}")
print(f"  CatBoost: {best_weights[1]:.4f}")
print(f"  NeuralNet:{best_weights[2]:.4f}")

# Create the final weighted ensemble predictions using the optimal weights
oof_ensemble_mean = (best_weights[0] * oof_xgb_preds +
                     best_weights[1] * oof_catboost_preds +
                     best_weights[2] * oof_nn_preds)

test_ensemble_mean = (best_weights[0] * test_xgb_preds +
                      best_weights[1] * test_catboost_preds +
                      best_weights[2] * test_nn_preds)

ensemble_rmse = np.sqrt(mean_squared_error(y_true, oof_ensemble_mean))
print(f"\nFinal Optimized 3-Model Ensemble OOF RMSE: ${ensemble_rmse:,.2f}")


--- Starting Block 4: Finding Optimal Weights for the 3-Model Ensemble ---

Weight optimization complete.
Optimal Ensemble Weights:
  XGBoost:  0.4091
  CatBoost: 0.4886
  NeuralNet:0.1023

Final Optimized 3-Model Ensemble OOF RMSE: $94,957.93


In [18]:
# =============================================================================
# BLOCK 3.5: SAVE K-FOLD PREDICTION ARRAYS
# =============================================================================
print("\n--- Starting Block 3.5: Saving K-Fold Prediction Arrays ---")

# --- Define Save Path ---
# As requested, we'll create a new directory to store these valuable outputs.
SAVE_PATH = './kfold_oof_predictions/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Prediction arrays will be saved in: '{SAVE_PATH}'")

# --- Save the four key prediction arrays ---
try:
    # Define file paths
    oof_xgb_path = os.path.join(SAVE_PATH, 'oof_xgb_preds.npy')
    test_xgb_path = os.path.join(SAVE_PATH, 'test_xgb_preds.npy')
    oof_cb_path = os.path.join(SAVE_PATH, 'oof_catboost_preds.npy')
    test_cb_path = os.path.join(SAVE_PATH, 'test_catboost_preds.npy')

    # Save each array to a .npy file
    np.save(oof_xgb_path, oof_xgb_preds)
    print(f"Saved: {oof_xgb_path}")
    
    np.save(test_xgb_path, test_xgb_preds)
    print(f"Saved: {test_xgb_path}")

    np.save(oof_cb_path, oof_catboost_preds)
    print(f"Saved: {oof_cb_path}")

    np.save(test_cb_path, test_catboost_preds)
    print(f"Saved: {test_cb_path}")

    print("\nAll K-Fold prediction arrays have been successfully saved.")
    
except NameError:
    print("\nERROR: One or more prediction arrays were not defined. Please ensure Block 3 ran correctly.")
except Exception as e:
    print(f"\nAn unexpected error occurred during saving: {e}")


--- Starting Block 3.5: Saving K-Fold Prediction Arrays ---
Prediction arrays will be saved in: './kfold_oof_predictions/'
Saved: ./kfold_oof_predictions/oof_xgb_preds.npy
Saved: ./kfold_oof_predictions/test_xgb_preds.npy
Saved: ./kfold_oof_predictions/oof_catboost_preds.npy
Saved: ./kfold_oof_predictions/test_catboost_preds.npy

All K-Fold prediction arrays have been successfully saved.


In [16]:
# =============================================================================
# BLOCK 5: TUNE & TRAIN THE ENSEMBLE ERROR MODEL
# =============================================================================
print("\n--- Starting Block 5: Tuning and Training the Ensemble Error Model ---")

# --- A. Define the Error Target and Feature Set ---
# The target is the absolute error of our optimized mean ensemble.
error_target = np.abs(y_true - oof_ensemble_mean)

# The features for the error model include the original features PLUS the mean ensemble's prediction
X_for_error = X.copy()
X_for_error['mean_ensemble_pred'] = oof_ensemble_mean
print("Error target and feature set created.")

# --- B. Tune the Error Model with Optuna ---
def objective_error_model(trial):
    dtrain = xgb.DMatrix(X_train_opt, label=y_train_opt)
    dval = xgb.DMatrix(X_val_opt, label=y_val_opt)
    params = { 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'n_jobs': -1, 'seed': RANDOM_STATE, 'eta': trial.suggest_float('eta', 0.01, 0.1, log=True), 'max_depth': trial.suggest_int('max_depth', 3, 8), 'subsample': trial.suggest_float('subsample', 0.5, 0.9), 'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9), 'lambda': trial.suggest_float('lambda', 1e-2, 20.0, log=True), 'alpha': trial.suggest_float('alpha', 1e-2, 20.0, log=True) }
    bst = xgb.train(params, dtrain, num_boost_round=2000, evals=[(dval, 'val')], early_stopping_rounds=75, verbose_eval=False)
    preds = bst.predict(dval, iteration_range=(0, bst.best_iteration))
    return np.sqrt(mean_squared_error(y_val_opt, preds))

X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(X_for_error, error_target, test_size=0.2, random_state=RANDOM_STATE)
study_error = optuna.create_study(direction='minimize')
print("\nTuning the XGBoost error model with Optuna...")
study_error.optimize(objective_error_model, n_trials=30)
best_params_error = study_error.best_params
print("Error model tuning complete.")

# --- C. K-Fold Train the Tuned Error Model ---
print("\nK-Fold training the final error model...")
oof_error_preds = np.zeros(len(X_for_error))
test_error_preds = np.zeros(len(X_test))
X_test_for_error = X_test.copy()
X_test_for_error['mean_ensemble_pred'] = test_ensemble_mean

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error, grade_for_stratify)):
    print(f"  Training error model fold {fold+1}/{N_SPLITS}...")
    dtrain_err = xgb.DMatrix(X_for_error.iloc[train_idx], label=error_target.iloc[train_idx])
    dval_err = xgb.DMatrix(X_for_error.iloc[val_idx])
    
    bst_error = xgb.train(best_params_error, dtrain_err, num_boost_round=3000, evals=[(xgb.DMatrix(X_for_error.iloc[val_idx], error_target.iloc[val_idx]), 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_error_preds[val_idx] = bst_error.predict(dval_err, iteration_range=(0, bst_error.best_iteration))
    test_error_preds += bst_error.predict(xgb.DMatrix(X_test_for_error), iteration_range=(0, bst_error.best_iteration)) / N_SPLITS

error_model_rmse = np.sqrt(mean_squared_error(error_target, oof_error_preds))
print(f"\nFinal Error Model OOF RMSE: ${error_model_rmse:,.2f}")


--- Starting Block 5: Tuning and Training the Ensemble Error Model ---
Error target and feature set created.


[I 2025-07-22 19:22:55,672] A new study created in memory with name: no-name-dff462ed-5db9-41a2-9dd0-a53b4b3bc3dc



Tuning the XGBoost error model with Optuna...


[I 2025-07-22 19:23:04,443] Trial 0 finished with value: 61769.90680321087 and parameters: {'eta': 0.034860739157679226, 'max_depth': 4, 'subsample': 0.5141590199386822, 'colsample_bytree': 0.8591765588853169, 'lambda': 0.02218151894333702, 'alpha': 4.361839616478466}. Best is trial 0 with value: 61769.90680321087.
[I 2025-07-22 19:23:11,015] Trial 1 finished with value: 61721.462091782916 and parameters: {'eta': 0.0536681292783415, 'max_depth': 5, 'subsample': 0.8268527078834016, 'colsample_bytree': 0.8671081886101628, 'lambda': 0.09980239469602002, 'alpha': 0.46678826953005365}. Best is trial 1 with value: 61721.462091782916.
[I 2025-07-22 19:23:28,703] Trial 2 finished with value: 61512.19541575189 and parameters: {'eta': 0.017595323369386153, 'max_depth': 6, 'subsample': 0.5828984727585192, 'colsample_bytree': 0.7968602686445326, 'lambda': 0.25085931973984615, 'alpha': 0.012770382268486553}. Best is trial 2 with value: 61512.19541575189.
[I 2025-07-22 19:23:50,431] Trial 3 finished

Error model tuning complete.

K-Fold training the final error model...
  Training error model fold 1/5...
  Training error model fold 2/5...
  Training error model fold 3/5...
  Training error model fold 4/5...
  Training error model fold 5/5...

Final Error Model OOF RMSE: $60,329.30


In [17]:
# =============================================================================
# BLOCK 6: FINAL CALIBRATION & SUBMISSION
# =============================================================================
print("\n--- Starting Block 6: Final Calibration for Winkler Score and Submission ---")

# Clip error predictions to be non-negative.
oof_error_final = np.clip(oof_error_preds, 0, None)
test_error_final = np.clip(test_error_preds, 0, None)

# Grid search for optimal Winkler Score multipliers 'a' and 'b'
best_score = float('inf')
best_a, best_b = 1.0, 1.0

print("\nStarting final calibration grid search to minimize Winkler score...")
for a in np.arange(1.8, 2.8, 0.01):
    for b in np.arange(1.8, 2.8, 0.01):
        low = oof_ensemble_mean - oof_error_final * a
        high = oof_ensemble_mean + oof_error_final * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score:
            best_score = score
            best_a, best_b = a, b

print("\n" + "="*60)
print("--- FINAL RESULTS ---")
print("="*60)
print(f"Best OOF Winkler Score: {best_score:,.2f}")
print(f"(This score is our best local estimate of leaderboard performance)")
print(f"Optimal Multipliers Found: a={best_a:.3f} (for lower bound), b={best_b:.3f} (for upper bound)")

# --- Create Submission File ---
print("\n--- Creating final submission file... ---")
final_lower = test_ensemble_mean - test_error_final * best_a
final_upper = test_ensemble_mean + test_error_final * best_b
# Ensure the lower bound never exceeds the upper bound.
final_upper = np.maximum(final_lower, final_upper)

# Load original test IDs to ensure correct submission format
submission_ids = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))['id']
submission_df = pd.DataFrame({'id': submission_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})

submission_filename = f'submission_3model_ensemble_winkler_{int(best_score)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
print("\nSubmission Head:")
print(submission_df.head())


--- Starting Block 6: Final Calibration for Winkler Score and Submission ---

Starting final calibration grid search to minimize Winkler score...

--- FINAL RESULTS ---
Best OOF Winkler Score: 292,284.38
(This score is our best local estimate of leaderboard performance)
Optimal Multipliers Found: a=1.970 (for lower bound), b=2.180 (for upper bound)

--- Creating final submission file... ---

'submission_3model_ensemble_winkler_292284.csv' created successfully! Good luck on the leaderboard!

Submission Head:
       id       pi_lower      pi_upper
0  200000  819174.234274  1.014099e+06
1  200001  569335.468727  7.865254e+05
2  200002  459659.466452  6.465838e+05
3  200003  294317.782559  4.217283e+05
4  200004  361241.874956  7.486215e+05


In [19]:
# =============================================================================
# BLOCK 5: TUNE & TRAIN THE ENSEMBLE ERROR MODEL (CORRECTED)
# =============================================================================
print("\n--- Starting Block 5: Tuning and Training the Ensemble Error Model ---")

# --- A. Define the Error Target and Feature Set ---
error_target = np.abs(y_true - oof_ensemble_mean)
X_for_error = X.copy()
X_for_error['mean_ensemble_pred'] = oof_ensemble_mean
print("Error target and feature set created for training.")

# (Optuna tuning part runs here as before...)
# ...
print("Error model tuning complete.")

# --- C. K-Fold Train the Tuned Error Model ---
print("\nK-Fold training the final error model...")
oof_error_preds = np.zeros(len(X_for_error))
test_error_preds = np.zeros(len(X_test))

# Create the base test set for the error model
X_test_for_error = X_test.copy()
X_test_for_error['mean_ensemble_pred'] = test_ensemble_mean

# === THE CRUCIAL FIX IS HERE ===
# Ensure the column order of the test set EXACTLY matches the training set.
# This prevents the model from misinterpreting the features.
X_test_for_error = X_test_for_error[X_for_error.columns]
print("CRITICAL FIX APPLIED: Ensured test feature order matches train feature order for the error model.")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error, grade_for_stratify)):
    print(f"  Training error model fold {fold+1}/{N_SPLITS}...")
    dtrain_err = xgb.DMatrix(X_for_error.iloc[train_idx], label=error_target.iloc[train_idx])
    dval_err = xgb.DMatrix(X_for_error.iloc[val_idx]) # DMatrix for validation
    
    # Train the model
    bst_error = xgb.train(best_params_error, dtrain_err, num_boost_round=3000, evals=[(xgb.DMatrix(X_for_error.iloc[val_idx], error_target.iloc[val_idx]), 'val')], early_stopping_rounds=100, verbose_eval=False)
    
    # Predict on the OOF set
    oof_error_preds[val_idx] = bst_error.predict(dval_err, iteration_range=(0, bst_error.best_iteration))
    
    # Predict on the FULL (and correctly ordered) test set
    dtest_err = xgb.DMatrix(X_test_for_error)
    test_error_preds += bst_error.predict(dtest_err, iteration_range=(0, bst_error.best_iteration)) / N_SPLITS

error_model_rmse = np.sqrt(mean_squared_error(error_target, oof_error_preds))
print(f"\nFinal Error Model OOF RMSE: ${error_model_rmse:,.2f}")


--- Starting Block 5: Tuning and Training the Ensemble Error Model ---
Error target and feature set created for training.
Error model tuning complete.

K-Fold training the final error model...
CRITICAL FIX APPLIED: Ensured test feature order matches train feature order for the error model.
  Training error model fold 1/5...
  Training error model fold 2/5...
  Training error model fold 3/5...
  Training error model fold 4/5...
  Training error model fold 5/5...

Final Error Model OOF RMSE: $60,329.30


In [20]:
# =============================================================================
# BLOCK 6: FORENSIC ANALYSIS & FINAL SUBMISSION (CORRECTED)
# =============================================================================
print("\n--- Starting Block 6: Final Analysis, Calibration, and Submission ---")

# --- Step 1: Clip Error Predictions ---
# Ensure that our predicted errors are never negative. This is a crucial sanity check.
oof_error_final = np.clip(oof_error_preds, 0, None)
test_error_final = np.clip(test_error_preds, 0, None)
print("Step 1: Final error predictions clipped to be non-negative.")

# --- Step 2: Forensic Analysis of Prediction Arrays ---
# Before we proceed, let's inspect the arrays that will create our submission file.
# We are checking for NaN (Not a Number), 'inf' (infinity), or extreme values
# which are the hallmarks of a data pipeline bug.
print("\n--- Step 2: Running Forensic Analysis on Test Set Predictions ---")
print("\nDescribing 'test_ensemble_mean' (our average price prediction):")
print(pd.Series(test_ensemble_mean).describe())

print("\nDescribing 'test_error_final' (our predicted error margin):")
print(pd.Series(test_error_final).describe())

if np.isnan(test_ensemble_mean).any() or np.isnan(test_error_final).any():
    print("\n\n*** CRITICAL WARNING: NaN values detected in final predictions! Submission will be invalid. ***\n")
else:
    print("\nSUCCESS: No NaN values detected in final prediction arrays.")
print("-------------------------------------------------------------------\n")

# --- Step 3: Calibrate Prediction Intervals using Winkler Score ---
# We search for the optimal multipliers 'a' and 'b' to create the tightest
# possible prediction intervals that still capture the true values, as measured
# by the competition's Winkler Score.
print("--- Step 3: Starting final calibration grid search... ---")
best_score = float('inf')
best_a, best_b = 1.0, 1.0

for a in np.arange(1.8, 2.8, 0.01):
    for b in np.arange(1.8, 2.8, 0.01):
        low = oof_ensemble_mean - oof_error_final * a
        high = oof_ensemble_mean + oof_error_final * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score:
            best_score = score
            best_a, best_b = a, b

# --- Step 4: Create and Verify the Final Submission DataFrame ---
print("\n--- Step 4: Creating and Verifying the Submission File ---")
final_lower = test_ensemble_mean - test_error_final * best_a
final_upper = test_ensemble_mean + test_error_final * best_b
final_upper = np.maximum(final_lower, final_upper) # Final check to ensure upper > lower

# Load original test IDs to ensure correct submission format
submission_ids = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))['id']
submission_df = pd.DataFrame({'id': submission_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})

# Final diagnostic check on the submission DataFrame itself
print("\nDescribing the final submission DataFrame before saving:")
print(submission_df.describe())

if submission_df.isnull().sum().any():
    print("\n\n*** CRITICAL WARNING: Null values detected in the final submission DataFrame! ***\n")
else:
    print("\nSUCCESS: Final submission DataFrame is clean and ready.")


# --- Step 5: Save Submission and Print Final Results ---
print("\n" + "="*60)
print("--- FINAL RESULTS & SUBMISSION ---")
print("="*60)
print(f"Best OOF Winkler Score: {best_score:,.2f}")
print(f"This score is our best local estimate of leaderboard performance.")
print(f"Optimal Multipliers Found: a={best_a:.3f} (for lower bound), b={best_b:.3f} (for upper bound)")

submission_filename = f'submission_3model_ensemble_winkler_{int(best_score)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
print("\nSubmission Head:")
print(submission_df.head())


--- Starting Block 6: Final Analysis, Calibration, and Submission ---
Step 1: Final error predictions clipped to be non-negative.

--- Step 2: Running Forensic Analysis on Test Set Predictions ---

Describing 'test_ensemble_mean' (our average price prediction):
count    2.000000e+05
mean    -8.652765e+06
std      4.134777e+09
min     -1.849128e+12
25%      3.134251e+05
50%      4.716368e+05
75%      7.320310e+05
max      2.998222e+06
dtype: float64

Describing 'test_error_final' (our predicted error margin):
count    200000.000000
mean      53100.677547
std       53160.507224
min        5738.156860
25%       22351.737183
50%       35605.828613
75%       61025.229492
max      782190.937500
dtype: float64

SUCCESS: No NaN values detected in final prediction arrays.
-------------------------------------------------------------------

--- Step 3: Starting final calibration grid search... ---

--- Step 4: Creating and Verifying the Submission File ---

Describing the final submission DataF